
# Rookie Projection Models

This notebook builds statistical comps for incoming rookies using real college production, athletic testing from the NFL Combine, and historical rookie transition rates. Preseason usage from exhibition games is incorporated to produce initial Boom/Bust tiers.


In [ ]:

import pandas as pd
import numpy as np
import requests
import io
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors


### Load datasets

In [ ]:

# Load combine data
combine_url = 'https://github.com/nflverse/nflverse-data/releases/download/combine/combine.csv'
combine = pd.read_csv(io.StringIO(requests.get(combine_url, verify=False).text))

# Load snap count data for 2023 (historical rookies) and 2024 preseason usage
snap23_url = 'https://github.com/nflverse/nflverse-data/releases/download/snap_counts/snap_counts_2023.csv'
snap24_url = 'https://github.com/nflverse/nflverse-data/releases/download/snap_counts/snap_counts_2024.csv'
snap23 = pd.read_csv(io.StringIO(requests.get(snap23_url, verify=False).text))
snap24 = pd.read_csv(io.StringIO(requests.get(snap24_url, verify=False).text))

# Load 2023 college player stats and aggregate rushing/receiving production
college_url = 'https://raw.githubusercontent.com/sportsdataverse/cfbfastR-data/main/player_stats/csv/player_stats_2023.csv'
college = pd.read_csv(io.StringIO(requests.get(college_url, verify=False).text))

rush = college.dropna(subset=['rush_player']).groupby('rush_player')['rush_yds'].sum()
rec = college.dropna(subset=['reception_player']).groupby('reception_player')['reception_yds'].sum()
td = college.dropna(subset=['touchdown_player']).groupby('touchdown_player').size()
college_prod = pd.concat([rush, rec, td], axis=1).fillna(0)
college_prod.columns = ['rush_yds','rec_yds','tds']
college_prod['total_yards'] = college_prod['rush_yds'] + college_prod['rec_yds']
college_prod = college_prod.reset_index().rename(columns={'index':'player_name'})


### Build historical and rookie datasets

In [ ]:

# Historical rookies from 2023 draft
hist = combine[combine['draft_year']==2023][['player_name','pos','forty','bench']]
reg_usage = snap23[snap23['game_type']=='REG'].groupby('player')['offense_snaps'].sum().reset_index()
hist = hist.merge(reg_usage, left_on='player_name', right_on='player', how='left').fillna({'offense_snaps':0})
transition_rates = hist.groupby('pos')['offense_snaps'].apply(lambda x: (x>100).mean()).reset_index(name='p_above_100')

# Incoming rookies for 2024 season
rookies = combine[combine['draft_year']==2024][['player_name','pos','forty','bench']]
rookies = rookies.merge(college_prod[['player_name','total_yards','tds']], on='player_name', how='left')
pre_usage = snap24[snap24['game_type']=='PRE'].groupby('player')['offense_pct'].mean().reset_index(name='preseason_usage')
rookies = rookies.merge(pre_usage, left_on='player_name', right_on='player', how='left').fillna({'preseason_usage':0})


### Find statistical comparables

In [ ]:

features = ['forty','bench','total_yards','tds']
X_hist = hist[features].fillna(0)
X_rook = rookies[features].fillna(0)
scaler = StandardScaler()
X_hist_scaled = scaler.fit_transform(X_hist)
X_rook_scaled = scaler.transform(X_rook)

nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_hist_scaled)

_, indices = nn.kneighbors(X_rook_scaled)
rookies['comps'] = [hist.iloc[idx]['player_name'].tolist() for idx in indices]


### Integrate transition rates and preseason usage

In [ ]:

rookies = rookies.merge(transition_rates, on='pos', how='left')
rookies['mean_comp_usage'] = [hist.iloc[idx]['offense_snaps'].mean() for idx in indices]
rookies['boom_score'] = rookies['mean_comp_usage'] * (0.5 + rookies['preseason_usage'])
rookies['bust_score'] = rookies['mean_comp_usage'] * (1 - rookies['preseason_usage'])
conditions = [
    rookies['boom_score'] > rookies['mean_comp_usage']*1.1,
    rookies['bust_score'] < rookies['mean_comp_usage']*0.9
]
choices = ['Boom','Bust']
rookies['tier'] = np.select(conditions, choices, default='Neutral')
rookies[['player_name','pos','preseason_usage','p_above_100','tier']].head()


### Summary

In [ ]:

rookies[['player_name','pos','tier','mean_comp_usage']].head()
